# Phân tích kết quả E1 / E2 / E3

Notebook đọc kết quả evaluation để xuất bảng metric, biểu đồ và ảnh lỗi.
E1/E2 được dùng để chọn kiến trúc cho thí nghiệm WCE (E3).
Khi có đủ ba kết quả, so sánh mIoU và các chỉ số mô phỏng phun để viết báo cáo.

Notebook này chạy trên CPU, không huấn luyện hoặc chạy inference.


In [ ]:
import csv
import sys
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'DO_AN_AI',
              Path('/kaggle/working/DO_AN_AI'), Path('/content/DO_AN_AI')]
PROJECT = next((p for p in candidates if (p / 'src/evaluation/analysis.py').is_file()), None)
if PROJECT is None and Path('/kaggle/input').is_dir():
    # Kaggle có thể giữ ZIP hoặc tự giải nén code Input.
    source_roots = sorted({p.parents[2] for p in Path('/kaggle/input').rglob('analysis.py')
                           if p.parent.name == 'evaluation' and p.parent.parent.name == 'src'})
    if len(source_roots) == 1:
        PROJECT = source_roots[0]
    elif not source_roots:
        archives = list(Path('/kaggle/input').rglob('DO_AN_AI_cloud.zip'))
        if len(archives) == 1:
            target = Path('/kaggle/working')
            with zipfile.ZipFile(archives[0]) as archive:
                for name in archive.namelist():
                    if not (target / name).resolve().is_relative_to(target.resolve()):
                        raise ValueError('Đường dẫn không hợp lệ trong ZIP: ' + name)
                archive.extractall(target)
            PROJECT = target / 'DO_AN_AI'
if PROJECT is None or not (PROJECT / 'src/evaluation/analysis.py').is_file():
    raise FileNotFoundError('Mở từ repo DO_AN_AI hoặc thêm đúng một bộ code ZIP vào Kaggle Input.')
WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/input').is_dir() else PROJECT
sys.path.insert(0, str(PROJECT))
from src.evaluation.analysis import (
    assert_comparable, choose_ce_experiment, load_report, summary_row,
)
print('Source:', PROJECT)


## 1. Chọn thư mục kết quả

Mỗi đường dẫn trong `RESULT_DIRS` phải trỏ tới thư mục chứa trực tiếp `metrics.json`,
ví dụ `results/E1/metrics.json`. Nếu giải nén ZIP thành `evaluation_E1/E1/metrics.json`,
đường dẫn cần kết thúc bằng `evaluation_E1/E1`.

Trên Kaggle, sao chép đường dẫn thư mục từ Input. `ANALYSIS_OUTPUT` phải nằm
trong `/kaggle/working`. `TRAIN_LOGS` ánh xạ tên thí nghiệm sang file
`training_log.csv` nếu log không còn ở đường dẫn checkpoint ghi trong báo cáo.


In [ ]:
RESULT_DIRS = {
    'E1': PROJECT / 'results/E1',
    'E2': PROJECT / 'results/E2',
    'E3': PROJECT / 'results/E3',
}
ANALYSIS_OUTPUT = WORK_ROOT / 'results/analysis'
TRAIN_LOGS = {}


## 2. Đọc và kiểm tra báo cáo

Bảng so sánh nhận kết quả đánh giá đủ 772 ảnh validation và loại preview/smoke.
Các báo cáo cần cùng danh sách ảnh, mapping, normalization, chế độ inference
và cấu hình huấn luyện được kiểm tra trong `assert_comparable`.

Khi có khác biệt, notebook hiển thị lý do và dừng tạo bảng chung.
Phiên bản dataset và augmentation cần được xác nhận từ source và thông tin lần chạy.


In [ ]:
ANALYSIS_OUTPUT.mkdir(parents=True, exist_ok=True)
reports = {}
for name, folder in RESULT_DIRS.items():
    path = Path(folder) / 'metrics.json'
    if not path.is_file():
        print(f'{name}: không tìm thấy {path}')
        continue
    report = load_report(path)
    try:
        assert_comparable({name: report})
    except ValueError as error:
        print(f'{name}: loại khỏi bảng so sánh: {error}')
        continue
    reports[name] = report

COMPARABLE = False
if reports:
    try:
        assert_comparable(reports, equal_epoch_budget=True, equal_training_protocol=True)
        COMPARABLE = True
    except ValueError as error:
        print('Cần kiểm tra trước khi so sánh:', error)
rows = [summary_row(name, report) for name, report in reports.items()] if COMPARABLE else []
print('Báo cáo hợp lệ:', list(reports))


## 3. Bảng metric

Bảng hiển thị phần trăm; `comparison.csv` lưu tỉ lệ từ 0 đến 1.
Weed Coverage bằng Weed Recall. `best_epoch` là epoch có validation loss thấp nhất;
`training_budget_completed` xác nhận ngân sách huấn luyện từ log.


In [ ]:
columns = {
    'miou': 'mIoU',
    'crop_iou': 'Crop IoU',
    'weed_iou': 'Weed IoU',
    'weed_recall': 'Weed Recall',
    'crop_wrongly_sprayed_rate': 'Crop phun nhầm',
    'spray_area_ratio': 'Diện tích phun',
}
if rows:
    with (ANALYSIS_OUTPUT / 'comparison.csv').open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    lines = ['| Experiment | ' + ' | '.join(columns.values()) + ' |',
             '|---|' + '---|' * len(columns)]
    for row in rows:
        values = ['n/a' if row[key] is None else f'{row[key]:.2%}' for key in columns]
        lines.append('| ' + row['experiment'] + ' | ' + ' | '.join(values) + ' |')
    display(Markdown('\n'.join(lines)))
else:
    print('Chưa có báo cáo đủ điều kiện để tạo bảng chung.')


## 4. IoU từng lớp và chọn kiến trúc cho E3

So sánh IoU của Background, Crop và Weed riêng từng lớp.
Chọn giữa E1/E2 theo mIoU; hòa điểm thì dùng Weed IoU.
Bước chọn yêu cầu log xác nhận cả hai lần CE đã hoàn tất cùng ngân sách.

Kết quả chọn ở bước này xác định kiến trúc để huấn luyện E3 với WCE.
Model cuối được chọn sau khi so sánh cả E1/E2/E3 theo tiêu chí của dự án.


In [ ]:
if rows:
    x = np.arange(len(rows))
    width = 0.25
    fig, ax = plt.subplots(figsize=(7, 4))
    for offset, key, label in [(-1, 'background_iou', 'Background'),
                               (0, 'crop_iou', 'Crop'), (1, 'weed_iou', 'Weed')]:
        values = [np.nan if row[key] is None else row[key] for row in rows]
        ax.bar(x + offset * width, values, width, label=label)
    ax.set(xticks=x, xticklabels=[row['experiment'] for row in rows], ylim=(0, 1), ylabel='IoU')
    ax.legend()
    fig.tight_layout()
    fig.savefig(ANALYSIS_OUTPUT / 'per_class_iou.png', dpi=180)
    plt.show()


In [ ]:
try:
    selected = choose_ce_experiment(reports)
    model_name = reports[selected]['checkpoint']['model_name']
    print('Chọn cho E3:', selected, '/', model_name)
    print('Config:', f'configs/experiments/e3_{model_name}_wce.yaml')
    print('E3 train mới từ pretrained, không resume checkpoint CE.')
except ValueError as error:
    print('Chưa chọn E3:', error)


## 5. Mô phỏng phun

Đọc Weed Coverage cùng Crop Wrongly Sprayed Rate để đánh giá khả năng phủ cỏ
và mức phun nhầm cây trồng. Spray Area Ratio cho biết diện tích pixel được phun.
`1 - Spray Area Ratio` là phần diện tích pixel không phun, chưa đo lượng thuốc thực tế.


In [ ]:
if rows:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for row in rows:
        harm = row['crop_wrongly_sprayed_rate']
        coverage = row['weed_coverage']
        if harm is not None and coverage is not None:
            axes[0].scatter(harm, coverage, s=60)
            axes[0].annotate(row['experiment'], (harm, coverage), xytext=(5, 5), textcoords='offset points')
    axes[0].set(xlabel='Crop Wrongly Sprayed Rate (càng thấp càng tốt)',
                ylabel='Weed Coverage (càng cao càng tốt)', ylim=(0, 1))
    axes[0].grid(alpha=0.25)
    areas = [np.nan if row['spray_area_ratio'] is None else row['spray_area_ratio'] for row in rows]
    axes[1].bar([row['experiment'] for row in rows], areas)
    axes[1].set(ylabel='Spray Area Ratio', ylim=(0, 1))
    fig.tight_layout()
    fig.savefig(ANALYSIS_OUTPUT / 'spraying_metrics.png', dpi=180)
    plt.show()


## 6. Confusion matrix

Hàng là ground truth, cột là prediction. Mỗi ô ghi số pixel và tỉ lệ theo hàng.
Crop dự đoán thành Weed tương ứng phun nhầm; Weed dự đoán thành Background
hoặc Crop tương ứng bỏ sót.


In [ ]:
for name, report in reports.items():
    matrix = np.array(report['confusion_matrix'], dtype=np.int64)
    totals = matrix.sum(axis=1, keepdims=True)
    normalized = np.divide(matrix, totals, out=np.zeros_like(matrix, dtype=float), where=totals != 0)
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(normalized, vmin=0, vmax=1, cmap='Blues')
    for i in range(3):
        for j in range(3):
            color = 'white' if normalized[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{matrix[i, j]:,}\n{normalized[i, j]:.1%}',
                    ha='center', va='center', color=color, fontsize=8)
    ax.set(xticks=range(3), yticks=range(3), xticklabels=report['class_names'],
           yticklabels=report['class_names'], xlabel='Prediction', ylabel='Ground truth', title=name)
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(ANALYSIS_OUTPUT / f'confusion_{name}.png', dpi=180)
    plt.show()


## 7. Đường cong loss

`TRAIN_LOGS` ở bước 1 cho phép chỉ định file log sau khi chuyển kết quả từ cloud.
Validation loss tăng kéo dài trong khi train loss giảm là dấu hiệu cần kiểm tra overfit.
Không kết luận từ một vài epoch dao động hoặc so trực tiếp độ lớn CE với WCE.


In [ ]:
for name, report in reports.items():
    default_log = Path(report['checkpoint']['path']).parent / 'training_log.csv'
    log = Path(TRAIN_LOGS.get(name, default_log))
    if not log.is_file():
        print(f'{name}: điền TRAIN_LOGS để vẽ loss curve.')
        continue
    with log.open(newline='', encoding='utf-8') as file:
        log_rows = list(csv.DictReader(file))
    if not log_rows:
        continue
    epochs = [int(row['epoch']) for row in log_rows]
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(epochs, [float(row['train_loss']) for row in log_rows], label='train')
    ax.plot(epochs, [float(row['val_loss']) for row in log_rows], label='validation')
    ax.set(xlabel='Epoch', ylabel='Loss', title=name)
    ax.legend()
    fig.tight_layout()
    fig.savefig(ANALYSIS_OUTPUT / f'loss_{name}.png', dpi=180)
    plt.show()
    print(name, '| Epoch cuối trong log:', max(epochs),
          '| Ngân sách config:', report['config']['training']['epochs'])


## 8. Phân tích lỗi

`selected_cases.csv` chứa các ảnh được xếp hạng theo lỗi, không phải mẫu ngẫu nhiên
đại diện dataset. Đối chiếu RGB, ground truth, prediction, error map và spray mask
trong `figures/`, sau đó ghi `manual_category` và `manual_notes` vào bản CSV ở thư mục
có quyền ghi. Các nhóm cần xem gồm cỏ nhỏ, vùng tối, cây chồng nhau và vùng biên.
Nguyên nhân cần được xác nhận từ ảnh.

Báo cáo trình bày bảng metric, so sánh kiến trúc E1/E2, ảnh hưởng CE/WCE và
đánh đổi giữa phủ cỏ với phun nhầm cây trồng. Metric tổng hợp tính từ confusion matrix
cộng trên toàn tập validation. Ảnh test không có nhãn chỉ dùng để xem dự đoán local.


In [ ]:
for name in reports:
    folder = Path(RESULT_DIRS[name])
    path = folder / 'selected_cases.csv'
    if not path.is_file():
        continue
    with path.open(newline='', encoding='utf-8') as file:
        cases = list(csv.DictReader(file))
    print(f'{name}: {len(cases)} trường hợp lỗi | CSV: {path}')
    for case in cases[:3]:
        print(case['filename'], '| Lý do chọn:', case['selection_reason'],
              '| Weed GT:', case.get('gt_weed_pixels'), '| Crop GT:', case.get('gt_crop_pixels'))
        figure = folder / case['figure']
        if figure.is_file():
            display(Image(filename=str(figure), width=1000))
